In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 데이터셋 로딩, RoBERTa 학습, 성능 평가에 필요한 라이브러리를 설치합니다.
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
# 데이터 처리, 시각화, PyTorch 학습에 사용할 모듈을 불러옵니다.
import re
import ast
import json
import html
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from IPython.display import display, HTML
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

In [2]:
# 사용 가능한 GPU 또는 다른 장치를 선택해 모델과 입력 데이터를 같은 장치에 올립니다.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)

DEVICE: cpu


In [3]:
# 사전학습 RoBERTa의 Tokenizer를 불러와 글을 모델 입력으로 변환할 준비
MODEL_NAME = "klue/roberta-base"  # Hugging Face 모델 ID
MAX_LEN = 256                  # 한 번에 모델에 넣을 최대 토큰 수 (모델 한계 512)

# 모델과 함께 공개된 Tokenizer를 사용해야 토큰 ID가 맞음
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("모델:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocab 크기:", tokenizer.vocab_size)

c:\VSGit\Personal_Mini_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\이희승\.cache\huggingface\hub\models--klue--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


모델: klue/roberta-base
Tokenizer: BertTokenizer
Vocab 크기: 32000


In [ ]:
# # 셀 생략 가능
# # 짧은 예시 문장을 토큰과 숫자 ID로 바꾸어 Tokenizer 동작을 확인합니다.
# sample_text = "이 글이 사람의 답변인지 ChatGPT의 답변인지 판별합니다."

# tokens = tokenizer.tokenize(sample_text)   # 글을 토큰으로 분리
# token_ids = tokenizer.encode(sample_text)  # [CLS] ... [SEP]를 붙여 숫자 ID로 변환

# print("원문:", sample_text)
# print("토큰:", tokens)
# print("토큰 ID:", token_ids)
# print("특수 토큰:", tokenizer.convert_ids_to_tokens([token_ids[0], token_ids[-1]]))

In [ ]:
# # 셀 생략 가능
# # 예시 문장의 input_ids와 attention_mask 모양(.shape)을 확인
# # 이 셀을 실행하려면 바로 위 셀의 sample_text가 필요함
# sample_encoded = tokenizer(
#     [sample_text, "두 번째 예시 문장입니다."],
#     padding=True,                  # 두 문장의 길이를 맞춤
#     truncation=True,               # 최대 길이를 넘으면 자름
#     max_length=MAX_LEN,
#     return_token_type_ids=False,   # RoBERTa는 문장 구분 ID를 쓰지 않음
#     return_tensors="pt",           # PyTorch Tensor로 반환
# )

# # (B, L): 문장 2개, 가장 긴 문장 기준 길이
# print("input_ids:", sample_encoded["input_ids"].shape)
# print("attention_mask:", sample_encoded["attention_mask"].shape)

# print("\ninput_ids:")       # 토큰의 숫자 ID
# print(sample_encoded["input_ids"])

# print("\nattention_mask:")  # 실제 토큰 1, padding 0
# print(sample_encoded["attention_mask"])

# 데이터셋 준비

In [4]:
# Hugging Face Hub에서 한국어 HC3-ko 데이터셋을 불러옵니다.
DATASET_NAME = "nayohan/HC3-ko"  # Hugging Face 데이터셋 ID

raw_dataset = load_dataset(
    DATASET_NAME,
    split="train",  # train 분할 하나만 제공됨
)

print(raw_dataset)
print("전체 질문 행 수:", len(raw_dataset))

c:\VSGit\Personal_Mini_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\이희승\.cache\huggingface\hub\datasets--nayohan--HC3-ko. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\VSGit\Personal_Mini_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingfac

Dataset({
    features: ['question', 'human_answers', 'chatgpt_answers', 'id', 'source'],
    num_rows: 24322
})
전체 질문 행 수: 24322


In [5]:
# 셀 생략 가능
# 원본 컬럼과 예시 값을 확인해 전처리에 필요한 데이터 형태를 살펴봅니다.
print("컬럼:", raw_dataset.column_names)

sample_row = raw_dataset[0]

for key, value in sample_row.items():
    print(f"\n[{key}] type={type(value).__name__}")
    print(str(value)[:300])

컬럼: ['question', 'human_answers', 'chatgpt_answers', 'id', 'source']

[question] type=str
내가 듣는 모든 책이 "NY Times # 1 베스트셀러"인 이유는 무엇입니까? ELI5 : 내가 듣는 모든 책이 "NY Times # 1 베스트셀러"인 이유는 무엇입니까? " # 1" 베스트셀러는 하나만 있어야 하지 않을까요? 다섯 살짜리 아이처럼 설명해 주세요.

[human_answers] type=str
['기본적으로 "베스트셀러"에는 여러 가지 카테고리가 있습니다. "베스트셀러"를 "오스카"와 같은 것으로 바꾸면 기본적으로 모든 "베스트셀러" 도서는 "오스카 수상작" 도서입니다. "베스트 필름"에 당첨되지 않았더라도 "베스트 감독"이나 "베스트 각본"에 당첨되면 여전히 "오스카 수상작" 영화가 됩니다. 베스트셀러도 마찬가지입니다. 또한, 기억하기로는 순위가 매주 변경되는 것 같습니다. 한 주에 베스트셀러가 아닐 수도 있지만 다음 주에는 베스트셀러가 될 수도 있습니다. 오래 머물지 않더라도 여전히 그 지위를 달성한 것 같습니다. 따

[chatgpt_answers] type=str
['다양한 조직에서 발행하는 베스트셀러 목록에는 여러 가지가 있으며 뉴욕 타임즈는 그 중 하나에 불과합니다. 뉴욕 타임즈 베스트셀러 목록은 여러 소매업체의 판매 데이터를 기반으로 미국에서 가장 많이 팔린 책을 순위 매기는 주간 목록입니다. 이 목록은 뉴욕 타임즈 신문에 게재되며 도서 업계에서 가장 영향력 있는 베스트셀러 목록 중 하나로 널리 알려져 있습니다. \n뉴욕 타임즈 베스트셀러 목록이 유일한 베스트셀러 목록은 아니며, 다른 카테고리 또는 다른 국가에서 가장 많이 팔린 책을 순위 매기는 다른 목록도 많이 있다는 점에 유의하는 

[id] type=str
0

[source] type=str
reddit_eli5


### HC3-ko 데이터 구조
```
question        : 질문 1개
human_answers   : "['답변1', '답변2', ...]"  <- 리스트가 문자열로 저장됨
chatgpt_answers : "['답변1', ...]"           <- 리스트가 문자열로 저장됨
source          : reddit_eli5, open_qa, wiki_csai, finance, medicine 등 출처
```
- 답변 목록이 문자열이므로 `ast.literal_eval`로 실제 리스트로 바꿔야 함
- 질문 하나에서 사람 답변 1개 + ChatGPT 답변 1개를 뽑아 **클래스 균형(1:1)** 을 맞춤

In [6]:
# 문자열로 저장된 답변 목록을 실제 리스트로 바꾸어 답변만 추출합니다.
def parse_answers(value):
    # 데이터가 이미 목록으로 읽힌 경우
    if isinstance(value, list):
        answers = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return []

        if value.startswith("["):
            # "['답변1', '답변2']"처럼 저장된 문자열을 목록으로 변환
            try:
                answers = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []
        else:
            # 일반 문자열이면 답변 하나로 처리
            answers = [value]

    else:
        return []

    if not isinstance(answers, list):
        return []

    # 빈 답변과 문자열이 아닌 값은 제외
    return [
        answer.strip()
        for answer in answers
        if isinstance(answer, str) and answer.strip()
    ]

### 지름길(Shortcut) 제거
- HC3의 사람 답변(Reddit 등)에는 `[ 링크 글자 ] ( URL_0 )` 같은 자리표시자와 `단어 .`처럼 구두점 앞 공백이 남아 있음
- ChatGPT 답변에는 이런 흔적이 거의 없어서, 모델이 **문체가 아니라 이 흔적만 보고** 맞힐 수 있음
- 사람/ChatGPT 답변 **모두에 같은 정리**를 적용해 형식 차이로 판별하지 못하게 함

In [7]:
# 두 클래스 답변에 같은 규칙을 적용해 형식 흔적(URL 자리표시자, 구두점 공백)을 정리합니다.
def clean_text(text):
    text = html.unescape(text)

    # [ 링크 글자 ] ( URL_0 ) -> 링크 글자
    text = re.sub(r"\[\s*([^\]]*?)\s*\]\s*\(\s*URL_\d+\s*\)", r"\1", text)

    # 남은 URL 자리표시자 제거
    text = re.sub(r"\(?\s*URL_\d+\s*\)?", " ", text)

    # 구두점/닫는 괄호 앞 공백, 여는 괄호 뒤 공백 제거
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)

    # 연속된 공백을 하나로
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)

    return text.strip()


# 셀 생략 가능: 정리 전/후 비교
example = "출판사는 [ 적절한 서점에서 책 몇 권을 구입 ] ( URL_0 )하고 , 목록에 올립니다 ."
print("정리 전:", example)
print("정리 후:", clean_text(example))

정리 전: 출판사는 [ 적절한 서점에서 책 몇 권을 구입 ] ( URL_0 )하고 , 목록에 올립니다 .
정리 후: 출판사는 적절한 서점에서 책 몇 권을 구입하고, 목록에 올립니다.


In [ ]:
# 각 질문에서 사람 답변과 ChatGPT 답변을 한 개씩 골라 한 쌍으로 만듭니다.

# MIN_CHARS가 있는 코드는 주석처리 해도 됨
MIN_CHARS = 20  # 너무 짧은 답변은 판별 근거가 부족하므로 제외

pairs = []

for row_number, row in enumerate(raw_dataset):
    human_answers = [
        clean_text(answer)
        for answer in parse_answers(row["human_answers"])
    ]
    chatgpt_answers = [
        clean_text(answer)
        for answer in parse_answers(row["chatgpt_answers"])
    ]

    human_answers = [a for a in human_answers if len(a) >= MIN_CHARS]
    chatgpt_answers = [a for a in chatgpt_answers if len(a) >= MIN_CHARS]

    # 사람과 ChatGPT 답변이 모두 있는 질문만 사용
    if not human_answers or not chatgpt_answers:
        continue

    # 클래스 균형을 위해 두 클래스에서 첫 번째 답변 하나씩 선택
    pairs.append({
        "question": str(row["question"]).strip(),
        "human_text": human_answers[0],
        "chatgpt_text": chatgpt_answers[0],
        "row_number": row_number,
    })

pairs_df = pd.DataFrame(pairs)

print("사용 가능한 질문-답변 쌍:", len(pairs_df))

In [9]:
# Colab 학습 시간을 줄이기 위해 사용할 질문 쌍의 수를 제한
MAX_PAIRS = 4000  # 늘리면 성능은 오르지만 학습 시간도 늘어남

if len(pairs_df) > MAX_PAIRS:
    pairs_df = pairs_df.sample(n=MAX_PAIRS).reset_index(drop=True)

print("사용 질문 쌍:", len(pairs_df))
print("전체 답변 수:", len(pairs_df) * 2)

사용 질문 쌍: 4000
전체 답변 수: 8000


In [11]:
# 셀 생략 가능
# 글자 수와 토큰 수를 비교해 MAX_LEN가 적당한지 확인합니다.
pairs_df["human_length"] = pairs_df["human_text"].str.len()
pairs_df["chatgpt_length"] = pairs_df["chatgpt_text"].str.len()

display(
    pairs_df[["human_length", "chatgpt_length"]].describe()
)

for column, name in [("human_text", "사람"), ("chatgpt_text", "ChatGPT")]:
    token_lengths = np.array([
        len(ids)
        for ids in tokenizer(pairs_df[column].tolist())["input_ids"]
    ])

    print(
        f"{name:8s} 토큰 수 | "
        f"중앙값 {np.median(token_lengths):.0f} | "
        f"90% {np.percentile(token_lengths, 90):.0f} | "
        f"MAX_LEN({MAX_LEN}) 초과 {np.mean(token_lengths > MAX_LEN) * 100:.1f}%"
    )

,human_length,chatgpt_length
count,4000.000000,4000.0000
mean,326.138250,461.1250
std,389.068087,166.1092
min,20.000000,21.0000
25%,113.000000,348.0000
50%,217.000000,459.0000
75%,402.000000,556.0000
max,10007.000000,1818.0000


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (662 > 512). Running this sequence through the model will result in indexing errors


사람       토큰 수 | 중앙값 110 | 90% 342 | MAX_LEN(256) 초과 17.2%
ChatGPT  토큰 수 | 중앙값 228 | 90% 328 | MAX_LEN(256) 초과 34.4%


### 데이터 분할 (Train 80% / Validation 10% / Test 10%)
- ViT에서는 이미지를 무작위로 9:1 분할했지만, 여기서는 **같은 질문이 학습과 평가에 동시에 들어가지 않도록** 질문 단위(Group)로 분할
- 같은 질문의 답변이 양쪽에 있으면 질문 내용만으로 맞힐 수 있어 성능이 부풀려짐

In [22]:
# 같은 질문이 학습과 평가에 섞이지 않도록 질문 단위로 학습 데이터를 나눕니다.
splitter_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
)

train_idx, temp_idx = next(
    splitter_1.split(
        pairs_df,
        groups=pairs_df["question"],
    )
)

train_pairs = pairs_df.iloc[train_idx].reset_index(drop=True)
temp_pairs = pairs_df.iloc[temp_idx].reset_index(drop=True)

# 남은 질문을 다시 반씩 나누어 검증용과 최종 평가용으로 사용
splitter_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.5,
)

val_idx, test_idx = next(
    splitter_2.split(
        temp_pairs,
        groups=temp_pairs["question"],
    )
)

val_pairs = temp_pairs.iloc[val_idx].reset_index(drop=True)
test_pairs = temp_pairs.iloc[test_idx].reset_index(drop=True)

print("Train 질문 쌍:", len(train_pairs))
print("Val   질문 쌍:", len(val_pairs))
print("Test  질문 쌍:", len(test_pairs))

Train 질문 쌍: 3193
Val   질문 쌍: 409
Test  질문 쌍: 398


In [13]:
# 세 분할에 같은 질문이 중복되지 않는지 검사합니다.
train_questions = set(train_pairs["question"])
val_questions = set(val_pairs["question"])
test_questions = set(test_pairs["question"])

# 중복 질문이 있으면 평가가 과대 추정되므로 실행 중단
assert train_questions.isdisjoint(val_questions)
assert train_questions.isdisjoint(test_questions)
assert val_questions.isdisjoint(test_questions)

print("분할 사이에 겹치는 질문이 없습니다.")

분할 사이에 겹치는 질문이 없습니다.


In [ ]:
# 답변 쌍을 개별 글과 사람(0)/ChatGPT(1) 라벨로 바꿉니다.
LABEL_NAMES = ["HUMAN", "CHATGPT"]


def make_classification_df(pair_df):
    rows = []

    for _, row in pair_df.iterrows():
        rows.append({
            "text": row["human_text"],
            "label": 0,  # 사람 답변
        })
        rows.append({
            "text": row["chatgpt_text"],
            "label": 1,  # ChatGPT 답변
        })

    return pd.DataFrame(rows)


train_df = make_classification_df(train_pairs)
val_df = make_classification_df(val_pairs)
test_df = make_classification_df(test_pairs)

print("Train 글:", len(train_df))
print("Val   글:", len(val_df))
print("Test  글:", len(test_df))

In [15]:
# 셀 생략 가능
# 각 분할에 두 클래스가 1:1로 들어 있는지 확인합니다.
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    counts = part["label"].value_counts().sort_index()
    print(f"{name:10s} | HUMAN: {counts[0]} | CHATGPT: {counts[1]}")

Train      | HUMAN: 3196 | CHATGPT: 3196
Validation | HUMAN: 402 | CHATGPT: 402
Test       | HUMAN: 402 | CHATGPT: 402


# **3. Dataset / DataLoader**
### Dynamic Padding
- `padding="max_length"`는 모든 글을 256 토큰까지 채워서 짧은 글도 256 길이로 계산됨
- `DataCollatorWithPadding`은 **배치 안에서 가장 긴 글의 길이**까지만 채움
- Self-Attention 계산량은 길이의 제곱(O(L²))에 비례하므로 padding을 줄이면 학습 속도가 빨라짐
- ViT는 이미지 크기가 224 x 224로 고정이라 필요 없었지만, 글은 길이가 제각각이라 효과가 큼

In [16]:
# 글을 미리 토큰화해 두고, padding은 배치를 만들 때 수행하는 Dataset을 정의합니다.
class TextClassificationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.labels = dataframe["label"].tolist()

        # padding 없이 잘라내기만 함 (padding은 collator가 배치 단위로 수행)
        self.encoded = tokenizer(
            dataframe["text"].tolist(),
            truncation=True,
            max_length=max_length,
            return_token_type_ids=False,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.encoded["input_ids"][index],
            "attention_mask": self.encoded["attention_mask"][index],
            "labels": self.labels[index],
        }

In [17]:
# 학습/검증/평가 Dataset과 배치 단위 padding을 담당할 collator를 만듭니다.
train_dataset = TextClassificationDataset(train_df, tokenizer, MAX_LEN)
val_dataset = TextClassificationDataset(val_df, tokenizer, MAX_LEN)
test_dataset = TextClassificationDataset(test_df, tokenizer, MAX_LEN)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

Train: 6392
Val  : 804
Test : 804


In [18]:
# Dataset을 배치로 묶어 학습과 평가에 사용할 DataLoader를 만듭니다.
BATCH_SIZE = 16
NUM_WORKERS = 0  # 0이 아니면 노트북 환경에서 멈출 수 있음

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    collate_fn=collator,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,   # 학습할 때만 순서를 섞음
    **loader_kwargs,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print("Train batches:", len(train_loader))
print("Val   batches:", len(val_loader))
print("Test  batches:", len(test_loader))

Train batches: 400
Val   batches: 51
Test  batches: 51


In [19]:
# 셀 생략 가능
# 한 배치의 입력과 라벨 모양(.shape)을 확인합니다.
batch = next(iter(train_loader))

# (B, L): L은 배치마다 달라짐 (배치 안의 가장 긴 글 길이, 최대 MAX_LEN)
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)

input_ids: torch.Size([16, 256])
attention_mask: torch.Size([16, 256])
labels: torch.Size([16])


In [20]:
# 셀 생략 가능
# 실행하려면 위 배치 확인 셀에서 만든 batch가 필요합니다.
print("예시 labels:", batch["labels"][:8])

예시 labels: tensor([1, 0, 0, 0, 1, 0, 0, 1])


# **4. 모델 (RoBERTa + Classification Head)**
```
ViT 예제 : timm backbone(num_classes=0) + 직접 만든 Linear head 4개
RoBERTa  : AutoModelForSequenceClassification = roberta(backbone) + classifier(head)
```
- `num_labels=2`로 지정하면 사전학습 backbone 위에 새 classifier head가 붙음
- head는 무작위로 초기화되므로 `classifier.* MISSING`(newly initialized) 경고가, 사전학습용 `lm_head.*`는 `UNEXPECTED` 경고가 나오는 것이 정상

In [21]:
# 사전학습 RoBERTa에 2개 클래스용 분류 헤드를 붙여 모델을 준비합니다.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={i: name for i, name in enumerate(LABEL_NAMES)},
    label2id={name: i for i, name in enumerate(LABEL_NAMES)},
).to(DEVICE)

model

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 852.73it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [ ]:
# 셀 생략 가능
# 학습되는 파라미터 수를 확인합니다.
total_params = sum(p.numel() for p in model.parameters())
head_params = sum(p.numel() for p in model.classifier.parameters())

print(f"전체 파라미터: {total_params / 1e6:.1f}M")
print(f"Classifier head: {head_params / 1e6:.2f}M")

In [ ]:
# 셀 생략 가능
# 샘플 배치 2개로 backbone -> [CLS] -> head 순서의 모양(.shape)을 확인합니다.
# 실행하려면 위 배치 확인 셀에서 만든 batch가 필요합니다.
sample_input = {
    "input_ids": batch["input_ids"][:2].to(DEVICE),
    "attention_mask": batch["attention_mask"][:2].to(DEVICE),
}

model.eval()

with torch.no_grad():
    hidden = model.roberta(**sample_input).last_hidden_state  # backbone
    cls_vector = hidden[:, 0]                                 # [CLS] 토큰
    logits = model.classifier(hidden)                         # head (내부에서 [:, 0] 사용)

print("input_ids:      ", sample_input["input_ids"].shape)  # (B, L)
print("Encoder 출력:   ", hidden.shape)                      # (B, L, 768)
print("[CLS] 벡터:     ", cls_vector.shape)                  # (B, 768)
print("분류 logits:    ", logits.shape)                      # (B, 2)
print("ChatGPT 확률:   ", torch.softmax(logits, dim=1)[:, 1])

# **5. 학습 설정**
### Fine-tuning 하이퍼파라미터
- RoBERTa-base 미세조정 권장 범위: LR `1e-5 ~ 5e-5`, Epoch `2 ~ 4`, Warmup `6 ~ 10%`
- `bias`와 `LayerNorm` 가중치에는 weight decay를 적용하지 않음 (BERT 계열 표준 설정)
- ViT 예제는 Epoch마다 `CosineAnnealingLR.step()`을 했지만, 여기서는 **Warmup + Linear Decay를 배치마다** `step()`
    - 사전학습 가중치는 처음에 큰 LR을 받으면 망가지기 쉬워서 Warmup으로 LR을 천천히 올림
- 손실 함수는 모델 내부의 CrossEntropyLoss 사용 (`labels`를 넣으면 `outputs.loss` 반환)

In [ ]:
# Optimizer와 학습률 Scheduler를 만들어 Fine-tuning을 준비합니다.
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

# bias, LayerNorm은 weight decay에서 제외
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print("Total steps :", total_steps)
print("Warmup steps:", warmup_steps)
print(optimizer)

### AMP(Automatic Mixed Precision)
- GPU에서는 mixed precision을 사용하며 연산 속도를 높이고 GPU 메모리 사용량을 줄일 수 있음
- 자동 혼합 정밀도
    - FP32(32비트 부동소수점): 정밀한 계산. 메모리를 더 많이 사용. 계산량이 큼
    - FP16(16비트 부동소수점): 낮은 정밀도. 메모리를 절약. 계산량이 적음
> 상황에 맞게 섞어 사용할 수 있는 기술이 Mixed Precision

In [ ]:
use_amp = (DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print("AMP enabled: ", use_amp)

### 평가 지표
- Positive 클래스 = **ChatGPT(1)**
- **Precision**: ChatGPT라고 판정한 글 중 실제 ChatGPT 비율 → 낮으면 사람 글을 AI로 오판
- **Recall**: 실제 ChatGPT 글 중 잡아낸 비율 → 낮으면 AI 글을 놓침
- **F1**: Precision과 Recall의 조화 평균 → **Best model 선택 기준**
- **ROC-AUC**: 임계값(0.5)과 상관없이 ChatGPT 확률 점수가 두 클래스를 얼마나 잘 나누는지

In [ ]:
# 정답, 예측, ChatGPT 확률로 분류 지표를 계산합니다.
def compute_metrics(labels, predictions, scores):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "auc": roc_auc_score(labels, scores),
    }

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_predictions = []
    all_scores = []

    for batch in loader:
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(**batch)

        logits = outputs.logits.float()
        scores = torch.softmax(logits, dim=1)[:, 1]  # ChatGPT(1) 확률
        predictions = logits.argmax(dim=1)

        batch_size = batch["labels"].size(0)
        total_loss += outputs.loss.item() * batch_size
        total_samples += batch_size

        all_labels.extend(batch["labels"].cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_scores.extend(scores.cpu().tolist())

    return {
        "loss": total_loss / total_samples,
        **compute_metrics(all_labels, all_predictions, all_scores),
        "predictions": all_predictions,
        "scores": all_scores,
    }

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # Warmup/Linear scheduler는 배치마다 갱신

        batch_size = batch["labels"].size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

# **6. 학습**

In [ ]:
# 학습 결과와 Best model을 저장할 Google Drive 경로를 준비합니다.
MODEL_DIR = Path("/content/drive/MyDrive/HC3_ko")
BEST_MODEL_DIR = MODEL_DIR / "best_model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("결과 저장 위치:", MODEL_DIR)

In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_auc": [],
}

best_f1 = -1.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        scaler,
        DEVICE,
    )

    metrics = evaluate(
        model,
        val_loader,
        DEVICE,
    )

    history["train_loss"].append(train_loss)
    for key in ["loss", "accuracy", "precision", "recall", "f1", "auc"]:
        history[f"val_{key}"].append(metrics[key])

    print(
        f"[{epoch:02d}/{EPOCHS}] "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={metrics['loss']:.4f} | "
        f"Acc={metrics['accuracy']*100:6.2f}% | "
        f"P={metrics['precision']:.4f} | "
        f"R={metrics['recall']:.4f} | "
        f"F1={metrics['f1']:.4f} | "
        f"AUC={metrics['auc']:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e} | "
        f"{time.time()-start:.1f}s"
    )

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_epoch = epoch

        # Hugging Face 형식으로 모델, Tokenizer를 함께 저장
        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)

        print(f"  -> Best model 저장: {BEST_MODEL_DIR}")

print(f"\nBest Validation F1: {best_f1:.4f} (epoch {best_epoch})")

In [ ]:
# Epoch별 기록을 표로 정리합니다. (CSV 저장에서 사용하므로 실행 필요)
history_df = pd.DataFrame(history)
history_df.insert(0, "epoch", range(1, len(history_df) + 1))

display(history_df)

In [ ]:
epochs = history_df["epoch"]

plt.figure(figsize=(7, 4))
plt.plot(epochs, history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, history_df["val_loss"], marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs, history_df["val_f1"], marker="o", label="Validation F1")
plt.plot(epochs, history_df["val_auc"], marker="o", label="Validation ROC-AUC")
plt.plot(epochs, history_df["val_precision"], marker="o", linestyle="--", label="Validation Precision")
plt.plot(epochs, history_df["val_recall"], marker="o", linestyle="--", label="Validation Recall")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Performance (Positive = ChatGPT)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# **7. Test 평가**
- 마지막 Epoch가 아니라 **Validation F1이 가장 높았던** 모델을 불러와 평가
- Test 데이터는 학습과 모델 선택에 한 번도 사용하지 않은 질문들

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
best_model.eval()

print("Best epoch:", best_epoch)
print("Best validation F1:", f"{best_f1:.4f}")
print("클래스:", best_model.config.id2label)

In [ ]:
test_result = evaluate(best_model, test_loader, DEVICE)

print(f"Accuracy : {test_result['accuracy']:.4f}")
print(f"Precision: {test_result['precision']:.4f}")
print(f"Recall   : {test_result['recall']:.4f}")
print(f"F1-score : {test_result['f1']:.4f}")
print(f"ROC-AUC  : {test_result['auc']:.4f}")

In [ ]:
# 학습 기록과 Test 지표를 Drive에 저장해 나중에 다시 확인할 수 있게 합니다.
history_df.to_csv(
    MODEL_DIR / "training_history.csv",
    index=False,
    encoding="utf-8-sig",
)

results = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "max_pairs": MAX_PAIRS,
    "max_length": MAX_LEN,
    "epochs": EPOCHS,
    "lr": LR,
    "best_epoch": best_epoch,
    "best_validation_f1": best_f1,
    **{f"test_{key}": test_result[key] for key in ["accuracy", "precision", "recall", "f1", "auc"]},
}

with open(MODEL_DIR / "test_metrics.json", "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

print("결과 저장 완료:", MODEL_DIR)

In [ ]:
# ViT의 show_predictions처럼 Test 글의 정답과 예측을 함께 확인합니다.
# test_loader는 shuffle=False이므로 예측 결과의 순서가 test_df와 같습니다.
test_view = test_df.copy()
test_view["pred"] = test_result["predictions"]
test_view["score"] = test_result["scores"]

def show_predictions(view_df, max_items=8, only_wrong=False):
    rows = view_df
    if only_wrong:
        # 확신했는데 틀린 글부터 표시
        rows = rows[rows["label"] != rows["pred"]]
        rows = rows.assign(conf=(rows["score"] - 0.5).abs()).sort_values("conf", ascending=False)
    else:
        rows = rows.sample(n=min(max_items, len(rows)))

    blocks = []
    for _, row in rows.head(max_items).iterrows():
        correct = row["label"] == row["pred"]
        color = "#e8f3e8" if correct else "#ffe0e0"
        blocks.append(
            f'<div style="background:{color}; padding:8px; margin:5px 0; border-radius:5px;">'
            f'<b>GT: {LABEL_NAMES[row["label"]]} / Pred: {LABEL_NAMES[row["pred"]]} '
            f'(ChatGPT {row["score"] * 100:.1f}%)</b><br>'
            f'{html.escape(row["text"][:300])}{"..." if len(row["text"]) > 300 else ""}'
            f'</div>'
        )

    display(HTML("".join(blocks)))


show_predictions(test_view)

In [ ]:
# 셀 생략 가능
# 모델이 확신했는데 틀린 글을 보고 어떤 글을 헷갈리는지 확인합니다.
show_predictions(test_view, only_wrong=True)

# **8. 직접 판별해 보기**
- 전체 글의 ChatGPT 확률을 계산하고, 문장마다 앞뒤 문장을 함께 넣어 부분 점수를 표시
- 점수는 **HC3-ko 데이터 기준**의 판별 결과이므로 다른 AI 모델이나 다른 분야의 글에서는 정확도가 떨어질 수 있음

In [ ]:
# 글 여러 개를 한 번에 토큰화해 ChatGPT(1) 확률을 계산합니다.
@torch.no_grad()
def predict_scores(texts):
    encoded = best_tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding=True,
        return_token_type_ids=False,
        return_tensors="pt",
    ).to(DEVICE)

    logits = best_model(**encoded).logits.float()
    return torch.softmax(logits, dim=1)[:, 1].cpu().tolist()


def split_sentences(text):
    # 마침표, 물음표, 느낌표 뒤 공백 또는 줄바꿈 기준으로 문장 분리
    pieces = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [piece.strip() for piece in pieces if piece.strip()]

In [ ]:
# 전체 글 판정과 문장별(앞뒤 문맥 포함) 점수를 화면에 표시합니다.
def show_detection(text):
    text = clean_text(text)  # 학습 데이터와 같은 정리 규칙 적용
    if not text:
        raise ValueError("검사할 글을 입력하세요.")

    score = predict_scores([text])[0]
    token_count = len(best_tokenizer(text)["input_ids"])

    print(f"HC3-ko 기준 ChatGPT 의심 점수: {score * 100:.1f}%")
    print("판정:", "ChatGPT 답변과 비슷함" if score >= 0.5 else "사람 답변과 비슷함")

    if token_count > MAX_LEN:
        print(f"전체 {token_count}토큰 중 앞 {MAX_LEN}토큰만 전체 점수에 사용했습니다.")

    sentences = split_sentences(text)
    if len(sentences) < 2:
        print("문장이 하나여서 문장별 표시는 생략합니다.")
        return

    # 각 문장과 앞뒤 문장을 묶어 한 번에 점수 계산
    contexts = [
        " ".join(sentences[max(0, i - 1): i + 2])
        for i in range(len(sentences))
    ]
    local_scores = predict_scores(contexts)

    print("\n문장별 의심 점수:")
    blocks = []
    for sentence, local_score in zip(sentences, local_scores):
        color = "#ffe0e0" if local_score >= 0.5 else "#e8f3e8"
        blocks.append(
            f'<div style="background:{color}; padding:8px; margin:5px 0; border-radius:5px;">'
            f'{html.escape(sentence)} <b>({local_score * 100:.1f}%)</b>'
            f'</div>'
        )

    display(HTML("".join(blocks)))

In [ ]:
# 직접 작성한 글을 넣어 판별 결과를 확인합니다.
my_text = """
여기에 직접 검사할 한국어 글을 입력하세요.
문장을 여러 개 입력하면 문장별 의심 점수도 함께 표시됩니다.
"""

show_detection(my_text)